# Training with extra features: weather, time, and location

This Source Code Form is subject to the terms of the Mozilla Public  License, v. 2.0. If a copy of the MPL was not distributed with this file, You can obtain one at http://mozilla.org/MPL/2.0/.
This notebook demonstrates how to train a model with additional contextual features:

* Weather data aligned to your time series locations  
* Location metadata (latitude and longitude)  
* Time based features provided by the configured dataset loader 

The goal is to show how these features are wired into the pipeline.

# Prerequisites
- You have run 00_data_preparation.ipynb to create the synthetic dataset.

In [ ]:
import json
import pprint
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import base64
import pathlib
from pathlib import Path

import folium
import pandas as pd
import tomlkit
import torch
from folium.plugins import FastMarkerCluster
from IPython.display import HTML, Image
from IPython.display import display as ipy_display

from s4casting.core.config import Configuration, DatasetConfiguration
from s4casting.data.preparation.dataset_formatter import DatasetFormatter
from s4casting.data.preparation.weather import WeatherDatasetFormatter
from scripts.train import train
from tests.utils import create_locations_file

print("✓ Imports successful!")

## Create locations file

This file has one row per location with `lon` and `lat`. It is used to:
* Fetch weather data for each location  
* Attach location metadata to the time series dataset so that weather and measurements can be aligned


In [ ]:
create_locations_file("data/locations")

pd.read_csv("data/locations/locations.csv")

## Build weather dataset

Next, we fetch and format weather features for the locations and date range of interest.

The output is a directory containing parquet files and a `weather.json` manifest. This manifest is referenced by the training configuration.

In [ ]:
WeatherDatasetFormatter(
    df_locations="data/locations/locations.csv",
    start_date="2023-01-01",
    end_date="2023-06-01",
    output_prefix="weather",
    output_dir="data/weather_output",
).run()

# pprint the json config
with Path("data/weather_output/weather.json").open("r", encoding="utf-8") as f:
    config = json.load(f)

pprint.pprint(config)

## Format measurements dataset with location metadata

The synthetic sinusoid measurements are wrapped into a dataset that includes location information.  
This allows the weather loader to align weather features with the measurement time series during training.


In [ ]:
DatasetFormatter(
    folder="data/sinusoid_data_raw",
    output_prefix="external_data_wrapped",
    output_dir="data/notebook_test_output_weather",
    target_col="measurements",
    time_col="timestamp",
    sample_interval_minutes=5,
    locations_file="data/locations/locations.csv",
).run()

with Path("data/notebook_test_output_weather/external_data_wrapped.json").open("r", encoding="utf-8") as f:
    config = json.load(f)

pprint.pprint(config)

## Build training configuration with weather features

We start from `configs/cuda-tiny.toml` and adapt it for this notebook:

* Point the `measurements` feature to the formatted sinusoid dataset  
* Add the `weather` dataset with a subset of features  
* Reduce the number of training steps and model size for quicker runs  
* Increase the learning rate to reach reasonable loss values in fewer steps  

In [ ]:
from s4casting.core.config import LocalBenchmark

with pathlib.Path("../configs/cuda-tiny.toml").open("r", encoding="utf-8") as f:
    config_data = tomlkit.load(f)

# Use smaller dataset for faster training in the notebook
config_data["io"]["features"]["measurements"]["location"] = (
    "data/notebook_test_output_weather/external_data_wrapped.json"
)
config_data["io"]["features"]["measurements"]["loader"] = "parquet"

config = Configuration(**config_data.unwrap())

# Use GPU if available
config.machine.device_kind = "cuda" if torch.cuda.is_available() else "cpu"

# Reduce number of training steps for faster training in the notebook
config.training.maximum_steps = 50
config.training.benchmarking_interval = 500
# Increase learning rate for faster convergence in the notebook
config.optimizer.learning_rate = 1e-2
# Reduce model size for faster training in the notebook
config.model.ssm.n_layers = 5
config.model.latent_dim = 128
# Reduce number of Gaussians for faster training in the notebook
config.model.output_head.n_gaussians = 2

config.io.feature_order = ["measurements", "weather"]
config.io.features["weather"] = DatasetConfiguration(
    location="data/weather_output/weather.json",
    loader="parquet",
    nearest_neighbor=True,
    main_dataset="measurements",
    n_features=4,
    subset_features=["temperature_2m", "wind_speed_100m", "shortwave_radiation", "direct_normal_irradiance"],
)

config.benchmarking.benchmarks["LocalBenchmark"] = LocalBenchmark(
    locations=["location_c"],
    thresholds=[8.0],
    n_day_ahead=1,
    context_window_days=config.model.context_window[0],
    predict_window_days=config.model.predict_width,
)

config

### Train with weather features

Train the model with the combined `measurements` and `weather` features.

> The measurements are synthetic, so training with real weather features here is not expected to yield meaningful better results. This section is primarily to visualize and validate the weather feature integration.

In [ ]:
train(config)

### Visualize benchmarking output with weather

The benchmarking script writes a forecast plot for each configured location.  
By default, this notebook expects a plot for `location_c` at step `500`.

In the plot, you can see the additional weather features (temperature_2m, wind_speed_100m, shortwave_radiation, direct_normal_irradiance) alongside the forecasts and actual values. The weather features are expected to help the model make better forecasts.

If you change the number of steps or benchmarking settings, adjust the file name accordingly.

In [ ]:
ipy_display(Image(filename=f"plots/benchmark_location_c_location_c_forecast_{config.training.maximum_steps}.png"))

## Training with additional contextual features

Beyond weather, the model can also use extra features such as:

* **Unix timestamp** (raw time index)
* **Location features** (longitude, latitude)

In the next example, we remove weather data but add time and location features to the training configuration.

In [ ]:
config.io.features.pop("weather", None)

config.io.feature_order = ["measurements", "time"]
config.io.features["time"] = DatasetConfiguration(
    location="",
    loader="time",
    main_dataset="measurements",
    n_features=3,
)
config

In [ ]:
train(config)

ipy_display(Image(filename=f"plots/benchmark_location_c_location_c_forecast_{config.training.maximum_steps}.png"))

## Sampling strategy for weather features

In real world scenarios, using weather for every single measurement point can be expensive.  
To keep the weather dataset manageable, we

* Sample a subset of locations within the service area  
* Fetch weather for these sampled locations  
* During training, assign each measurement location the weather of its nearest sampled point  

This reduces data volume while still approximating local weather conditions reasonably well.

The interactive map below visualizes the sampled weather locations together with measurement points.

### What you are looking at
- Blue markers: sampled weather coordinates used to fetch weather features.
- Cluster: measurement locations (clustered for performance).
- The map overlays both so you can see how sampled points cover the measurement area.

### How to explore the interactive map
- Pan and zoom with your mouse/trackpad; scroll to zoom.
- Click a blue marker to see its latitude/longitude.
- Click a cluster to zoom in; clusters split as you zoom until individual points are visible.

Note: To reproduce this visualization, make sure that `sampled_coordinates.csv` and `no_anomaly_measurements_no_oversampling.json` or an equivalent file are available.

In [ ]:
# Extract locations from Croissant measurements (data/LianderPower/measurements.parquet)
# Snap lat/lon to a GRID (0.25°) and build `sampled_coordinates` used for sampling weather points
from pathlib import Path

import numpy as np
import pandas as pd

# Grid snapping resolution in degrees
GRID = 0.25
release_dir = Path("../data/LianderPower")

# Load measurements using the Croissant adapter; this returns a NumpyData with a `locations` dict
from s4casting.data.dataset.croissant_adapter import load_measurements_from_croissant

ndata = load_measurements_from_croissant(release_dir, to_memory=False)
locs = ndata.locations

# Build a flat list of measurement locations and their snapped grid cell
rows = []
for lid, meta in locs.items():
    # meta is a mapping with keys like 'lat' and 'lon'
    lat = meta["lat"]
    lon = meta["lon"]
    # Snap coordinates to the GRID grid to produce ERA-like sampled points
    lat_s = float(np.round(lat / GRID) * GRID)
    lon_s = float(np.round(lon / GRID) * GRID)
    rows.append({
        "id": str(lid),
        "name": meta.get("name", "") if isinstance(meta, dict) else "",
        "lat": float(lat),
        "lon": float(lon),
        "snapped_lat": lat_s,
        "snapped_lon": lon_s,
    })

# Build a DataFrame and aggregate by snapped grid cell. The resulting
# `sampled_coordinates` DataFrame contains one row per snapped cell with
# merged original ids, optional human-readable names, mean coordinate, and count.
df = pd.DataFrame(rows)
sampled_coordinates = (
    df
    .groupby(["snapped_lat", "snapped_lon"])
    .agg(
        merged_ids=("id", list),
        names=("name", lambda x: list(dict.fromkeys([n for n in x if n]))),
        lat_mean=("lat", "mean"),
        lon_mean=("lon", "mean"),
        count=("id", "size"),
    )
    .reset_index()
)

print(f"Found {len(rows)} locations, {len(sampled_coordinates)} unique snapped cells in {release_dir}")
sampled_coordinates.head()

In [ ]:
# Create interactive map showing sampled coordinates (snapped grid) and measurement points
# This cell expects `sampled_coordinates` to exist in the notebook (produced above).
from pathlib import Path

import pandas as pd

from s4casting.data.dataset.croissant_adapter import load_measurements_from_croissant

# Load measurement locations directly from the Croissant release (these are the full measurement points)
release_dir = Path("../data/LianderPower")
ndata = load_measurements_from_croissant(release_dir, to_memory=False)
locs = ndata.locations
measurement_rows = [{"lat": float(m["lat"]), "lon": float(m["lon"])} for _, m in locs.items()]
df_measurements = pd.DataFrame(measurement_rows)

# Use the aggregated sampled coordinates (one row per snapped grid cell).
# The aggregation produced `lat_mean`/`lon_mean` columns; rename for plotting convenience.
df = sampled_coordinates.rename(columns={"lat_mean": "lat", "lon_mean": "lon"})

# Build the folium map centered on the sampled points
center = [float(df["lat"].mean()), float(df["lon"].mean())]
m = folium.Map(location=center, zoom_start=7, tiles="OpenStreetMap")

# Add blue markers for sampled coordinates (these are the weather sampling points)
for _, row in df.iterrows():
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"lat={row['lat']:.4f}, lon={row['lon']:.4f}",
        icon=folium.Icon(color="blue"),
    ).add_to(m)

# Add measurement points as a clustered red layer (many points)
coords = list(zip(df_measurements["lat"], df_measurements["lon"]))
FastMarkerCluster(coords, name="Measurements").add_to(m)

folium.LayerControl().add_to(m)

# Save to an HTML file and embed in the notebook for quick preview
out_html = Path("plots/sampled_locations_map.html").resolve()
out_html.parent.mkdir(parents=True, exist_ok=True)

m.save(out_html)
html_bytes = m.get_root().render().encode("utf-8")
data_url = "data:text/html;base64," + base64.b64encode(html_bytes).decode("ascii")

HTML(f"""
<iframe
    src="{data_url}"
    width="800"
    height="600"
    style="border: none;"
></iframe>
""")